# Analyse Tweets US Airline - Notebook

Ce notebook couvre:
1. Configuration de l environnement VS Code
2. Choix du kernel Python
3. Chargement et exploration du dataset
4. Preparation des donnees pour la classification de `airline_sentiment`
5. Visualisation et export

## 1) Configurer l environnement dans VS Code

- Extensions conseillees: Python, Jupyter
- Environnement virtuel deja utilise dans ce dossier
- Dependances minimales: `pandas`, `scikit-learn`, `matplotlib`

## 2) Creer et selectionner le kernel Python

Le kernel doit pointer vers l environnement virtuel local du projet.
La cellule suivante verifie la version Python et la presence des packages utiles.

In [ ]:
import sys
import platform
import importlib

print('Python version :', sys.version)
print('Platform       :', platform.platform())

for pkg in ['pandas', 'sklearn', 'matplotlib']:
    found = importlib.util.find_spec(pkg) is not None
    print(f'{pkg:10s} ->', 'OK' if found else 'MISSING')

## 3) Ecrire et executer des cellules de code

On charge le dataset puis on utilise les methodes pandas usuelles pour:
- dimensions
- types
- valeurs manquantes
- repartition de la cible

In [ ]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

DATA_PATH = 'Tweets (2).csv'
TARGET = 'airline_sentiment'

df = pd.read_csv(DATA_PATH)

print('Shape:', df.shape)
print('\nTypes:')
print(df.dtypes)
print('\nValeurs manquantes:')
print(df.isna().sum())
print('\nDistribution de la cible:')
print(df[TARGET].value_counts())

df.head()

In [ ]:
useful_variables = [
    'text',
    'airline',
    'airline_sentiment_confidence',
    'retweet_count',
    'user_timezone',
    'tweet_location',
    'tweet_created',
]

useful_variables = [c for c in useful_variables if c in df.columns]
print('Variables utiles possibles pour la classification de airline_sentiment:')
print(useful_variables)

## 4) Structurer avec variables, fonctions et tests rapides

On encode la cible `airline_sentiment` en entiers {0,1,2}, on split en train/validation et test, puis on construit un pipeline de pretraitement avec:
1. suppression des variables inutiles
2. fonction maison de nettoyage de tweets

In [ ]:
SENTIMENT_TO_INT = {'negative': 0, 'neutral': 1, 'positive': 2}

df_model = df.copy()
df_model[TARGET] = df_model[TARGET].map(SENTIMENT_TO_INT)
assert not df_model[TARGET].isna().any(), 'Certaines valeurs de la cible ne sont pas mappees.'
df_model[TARGET] = df_model[TARGET].astype(int)

def drop_useless_variables(dataframe: pd.DataFrame) -> pd.DataFrame:
    return dataframe[['text']].copy()

def clean_tweet_text(text: str) -> str:
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#', ' ', text)
    text = re.sub(r'&amp;', 'and', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_tweets(dataframe: pd.DataFrame) -> pd.DataFrame:
    out = dataframe.copy()
    out['text'] = out['text'].apply(clean_tweet_text)
    return out

X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val
)

pipeline = Pipeline(
    steps=[
        ('drop_useless', FunctionTransformer(drop_useless_variables, validate=False)),
        ('tweet_preprocess', FunctionTransformer(preprocess_tweets, validate=False)),
    ]
)

X_train_ready = pipeline.fit_transform(X_train)
X_val_ready = pipeline.transform(X_val)
X_test_ready = pipeline.transform(X_test)

print('Train+Val:', X_train_val.shape, ' Test:', X_test.shape)
print('Train:', X_train.shape, ' Val:', X_val.shape, ' Test:', X_test.shape)
print('\nExemple tweets preprocesses:')
X_train_ready.head()

In [ ]:
# Tests rapides
assert set(df_model[TARGET].unique()) == {0, 1, 2}
assert X_train_ready.columns.tolist() == ['text']
assert X_train_ready['text'].map(lambda x: isinstance(x, str)).all()
print('Assertions OK')

## 5) Visualiser les resultats et exporter le notebook

On affiche une visualisation simple de la distribution de `airline_sentiment` encodee.
Ensuite, dans VS Code:
- sauvegarder le notebook
- exporter via la commande `Jupyter: Export to Python Script` ou `Export to HTML`.

In [ ]:
import matplotlib.pyplot as plt

count_by_class = y.value_counts().sort_index()
label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
labels = [label_map[i] for i in count_by_class.index]

plt.figure(figsize=(7, 4))
plt.bar(labels, count_by_class.values)
plt.title('Distribution de airline_sentiment (encode)')
plt.xlabel('Classe')
plt.ylabel('Nombre de tweets')
plt.tight_layout()
plt.show()